# 🤖 OPAL Marketing — FAQ Chatbot
## CodeAlpha AI Internship | Task 2
### Built with NLP: TF-IDF + Cosine Similarity

In [1]:
# Import Libraries
import re
import math
from collections import Counter

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# FAQ Data for OPAL Marketing
FAQ_DATA = [
    {
        "question": "What is OPAL Marketing?",
        "answer": "OPAL Marketing is a leading Egyptian digital marketing agency specializing in social media management, content creation, branding, SEO, and paid advertising."
    },
    {
        "question": "What services does OPAL offer?",
        "answer": "OPAL offers Social Media Management, Content Creation, SEO, PPC Advertising, Brand Identity & Logo Design, Email Marketing, and Website Development."
    },
    {
        "question": "How much do your services cost?",
        "answer": "We offer flexible packages starting from 2,000 EGP/month for small businesses, up to custom enterprise plans. Contact us for a free quote."
    },
    {
        "question": "Do you offer free consultations?",
        "answer": "Yes! OPAL Marketing offers a free initial consultation to understand your business goals and recommend the best marketing strategy."
    },
    {
        "question": "How do I get started with OPAL?",
        "answer": "Reach out via our website or phone. We schedule a free consultation, propose a strategy, and kick off your campaign within 48 hours."
    },
    {
        "question": "Do you manage social media accounts?",
        "answer": "Yes! We handle content planning, post creation, scheduling, community management, and ad management on Facebook, Instagram, TikTok, and LinkedIn."
    },
    {
        "question": "Can you help with logo and branding?",
        "answer": "Yes! Our creative team designs logos, brand guidelines, and full brand identity packages tailored to your business."
    },
    {
        "question": "Do you provide reports and analytics?",
        "answer": "Yes! Every client receives detailed monthly reports including reach, engagement, leads, conversions, and ad spend breakdown."
    },
]

print(f"✅ Loaded {len(FAQ_DATA)} FAQs successfully!")

✅ Loaded 8 FAQs successfully!


In [3]:
# Text Preprocessing
# This function cleans and prepares text for comparison

STOPWORDS = {
    "a","an","the","is","it","in","on","of","to","and","or","for","with",
    "do","you","can","me","my","your","we","i","how","what","who","when",
    "where","does","are","was","were","be","been","have","has","this","that",
    "will","would","could","should","about","there","get","any","all","just"
}

def preprocess(text):
    text = text.lower()                          # كل الكلام بحروف صغيرة
    text = re.sub(r"[^a-z0-9\s]", "", text)     # شيل علامات الترقيم
    tokens = text.split()                        # قسم الجملة لكلمات
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # شيل الكلمات الزيادة
    return tokens

# Test it
print(preprocess("How much does it cost?"))
print(preprocess("Do you manage social media accounts?"))

['much', 'cost']
['manage', 'social', 'media', 'accounts']


In [4]:
# TF-IDF Vectorization
# TF  = كام مرة الكلمة اتكررت في الجملة دي
# IDF = الكلمة دي نادرة ولا شائعة في كل الأسئلة

def compute_tf(tokens):
    counts = Counter(tokens)
    total = len(tokens) if tokens else 1
    return {word: count/total for word, count in counts.items()}

def compute_idf(corpus):
    N = len(corpus)
    df = {}
    for doc in corpus:
        for word in set(doc):
            df[word] = df.get(word, 0) + 1
    return {word: math.log((N+1)/(df[word]+1))+1 for word in df}

# Build the index from all FAQs
corpus = [preprocess(f["question"] + " " + f["answer"]) for f in FAQ_DATA]
idf = compute_idf(corpus)
faq_vecs = []
for tokens in corpus:
    tf = compute_tf(tokens)
    vec = {word: tf[word] * idf.get(word, 1) for word in tf}
    faq_vecs.append(vec)

print(f"✅ Built TF-IDF vectors for {len(faq_vecs)} FAQs")
print(f"📊 Example - top words in FAQ 1: {sorted(faq_vecs[0].items(), key=lambda x: -x[1])[:5]}")

✅ Built TF-IDF vectors for 8 FAQs
📊 Example - top words in FAQ 1: [('marketing', 0.285936349928894), ('opal', 0.16713543841074938), ('leading', 0.13179354719875128), ('egyptian', 0.13179354719875128), ('digital', 0.13179354719875128)]


In [5]:
# Cosine Similarity
# بتقيس مدى التشابه بين سؤال اليوزر وكل سؤال في الـ FAQ
# النتيجة بين 0 و 1 — كلما اقتربت من 1 كلما كان التشابه أكبر

def cosine_similarity(vec1, vec2):
    common = set(vec1.keys()) & set(vec2.keys())  # الكلمات المشتركة
    dot = sum(vec1[w] * vec2[w] for w in common)  # حاصل الضرب
    norm1 = math.sqrt(sum(v**2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v**2 for v in vec2.values()))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

# Test it
q1 = "how much cost"
q2 = "social media management"
tokens1 = preprocess(q1)
tokens2 = preprocess(q2)
tf1 = compute_tf(tokens1)
tf2 = compute_tf(tokens2)
vec1 = {w: tf1[w] * idf.get(w, 1) for w in tf1}
vec2 = {w: tf2[w] * idf.get(w, 1) for w in tf2}

print(f"سؤال: '{q1}'")
for i, faq in enumerate(FAQ_DATA):
    score = cosine_similarity(vec1, faq_vecs[i])
    print(f"  {score:.3f} — {faq['question']}")

سؤال: 'how much cost'
  0.000 — What is OPAL Marketing?
  0.000 — What services does OPAL offer?
  0.329 — How much do your services cost?
  0.000 — Do you offer free consultations?
  0.000 — How do I get started with OPAL?
  0.000 — Do you manage social media accounts?
  0.000 — Can you help with logo and branding?
  0.000 — Do you provide reports and analytics?


In [6]:
# Main Chatbot Function
# دي الدالة الرئيسية اللي بتاخد سؤال اليوزر وترد عليه

def get_response(user_input):
    # Step 1: Preprocess the user question
    tokens = preprocess(user_input)
    
    if not tokens:
        return "Please ask me something about OPAL Marketing! 😊", 0
    
    # Step 2: Convert to TF-IDF vector
    tf = compute_tf(tokens)
    query_vec = {w: tf[w] * idf.get(w, 1) for w in tf}
    
    # Step 3: Compare with all FAQs
    scores = [cosine_similarity(query_vec, faq_vec) for faq_vec in faq_vecs]
    
    # Step 4: Get the best match
    best_idx = scores.index(max(scores))
    best_score = scores[best_idx]
    
    # Step 5: Return answer or fallback message
    if best_score < 0.2:
        return "I'm not sure about that! Contact us at info@opalmarketing.com 📞", best_score
    
    return FAQ_DATA[best_idx]["answer"], best_score

# Test it
questions = [
    "What services do you offer?",
    "How much does it cost?",
    "Do you manage social media?",
    "Can you help with branding?",
]

print("🤖 OPAL Chatbot Test\n" + "="*40)
for q in questions:
    answer, score = get_response(q)
    print(f"\n❓ {q}")
    print(f"✅ {answer[:80]}...")
    print(f"📊 Confidence: {round(score*100, 1)}%")

🤖 OPAL Chatbot Test

❓ What services do you offer?
✅ OPAL offers Social Media Management, Content Creation, SEO, PPC Advertising, Bra...
📊 Confidence: 28.7%

❓ How much does it cost?
✅ We offer flexible packages starting from 2,000 EGP/month for small businesses, u...
📊 Confidence: 32.9%

❓ Do you manage social media?
✅ Yes! We handle content planning, post creation, scheduling, community management...
📊 Confidence: 35.4%

❓ Can you help with branding?
✅ Yes! Our creative team designs logos, brand guidelines, and full brand identity ...
📊 Confidence: 33.2%


In [8]:
# Interactive Chat Interface
# دي الواجهة اللي اليوزر هيتكلم معاها

print("🤖 OPAL Marketing Chatbot")
print("=" * 40)
print("Type your question or 'quit' to exit\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == 'quit':
        print("Bot: Thank you for contacting OPAL Marketing! Goodbye! 👋")
        break
    
    if user_input.strip() == "":
        continue
    
    answer, score = get_response(user_input)
    print(f"Bot: {answer}")
    print(f"     (Confidence: {round(score*100, 1)}%)\n")

🤖 OPAL Marketing Chatbot
Type your question or 'quit' to exit

Bot: We offer flexible packages starting from 2,000 EGP/month for small businesses, up to custom enterprise plans. Contact us for a free quote.
     (Confidence: 32.9%)

Bot: Yes! We handle content planning, post creation, scheduling, community management, and ad management on Facebook, Instagram, TikTok, and LinkedIn.
     (Confidence: 41.4%)

Bot: I'm not sure about that! Contact us at info@opalmarketing.com 📞
     (Confidence: 15.1%)

Bot: Thank you for contacting OPAL Marketing! Goodbye! 👋
